Script 1: RPF Fix (run on ALL nodes first)

In [ ]:
cat > fix-rpf.sh <<OF
#!/bin/bash
# =============================================================================
# RPF Fix Script — Run this on ALL nodes before installing Calico
# Fixes: strict Reverse Path Filtering that drops pod return traffic
# Run on: master-1, master-2, master-3, worker-1, worker-2
# =============================================================================
set -euo pipefail

GREEN='\033[0;32m'; BLUE='\033[0;34m'; NC='\033[0m'
log()     { echo -e "${BLUE}[INFO]${NC} $1"; }
success() { echo -e "${GREEN}[SUCCESS]${NC} $1"; }

IFACE="${1:-ens192}"  # Pass interface name as argument if different, e.g. ./fix-rpf.sh eth0

log "Setting loose RPF (rp_filter=2) on interface: $IFACE"

# Apply immediately
sysctl -w net.ipv4.conf.all.rp_filter=2
sysctl -w net.ipv4.conf.default.rp_filter=2
sysctl -w net.ipv4.conf.${IFACE}.rp_filter=2

# Make permanent
cat > /etc/sysctl.d/99-kubernetes.conf << EOF
# Required for Kubernetes + Calico VXLAN + hostNetwork ingress
net.ipv4.conf.all.rp_filter = 2
net.ipv4.conf.default.rp_filter = 2
net.ipv4.conf.${IFACE}.rp_filter = 2

# Required for Kubernetes
net.bridge.bridge-nf-call-iptables = 1
net.bridge.bridge-nf-call-ip6tables = 1
net.ipv4.ip_forward = 1
EOF

sysctl -p /etc/sysctl.d/99-kubernetes.conf

success "RPF fix applied permanently on $(hostname)"
echo "Current rp_filter values:"
echo "  all:     $(cat /proc/sys/net/ipv4/conf/all/rp_filter)"
echo "  default: $(cat /proc/sys/net/ipv4/conf/default/rp_filter)"
echo "  ${IFACE}: $(cat /proc/sys/net/ipv4/conf/${IFACE}/rp_filter)"
OF

In [ ]:
# Copy fix-rpf.sh to each node and run it
chmod +x fix-rpf.sh && ./fix-rpf.sh ens192

---

Script 2: Full Cleanup

In [ ]:
cat > cleanup.sh <<'OOF'
#!/bin/bash
# =============================================================================
# Full Cleanup Script
# Removes: Headlamp, Ingress-NGINX, MetalLB, Calico
# =============================================================================
set -euo pipefail

RED='\033[0;31m'; GREEN='\033[0;32m'; YELLOW='\033[1;33m'; BLUE='\033[0;34m'; NC='\033[0m'
log()     { echo -e "${BLUE}[INFO]${NC} $1"; }
success() { echo -e "${GREEN}[SUCCESS]${NC} $1"; }
warn()    { echo -e "${YELLOW}[WARN]${NC} $1"; }

# ── 1. Headlamp ──────────────────────────────────────────────────────────────
log "Removing Headlamp..."
helm uninstall my-headlamp -n headlamp 2>/dev/null || warn "Headlamp not found"
kubectl delete namespace headlamp --ignore-not-found
success "Headlamp removed"

# ── 2. Ingress-NGINX ─────────────────────────────────────────────────────────
log "Removing Ingress-NGINX..."
helm uninstall ingress-nginx -n ingress-nginx 2>/dev/null || warn "ingress-nginx not found"
kubectl delete namespace ingress-nginx --ignore-not-found
success "Ingress-NGINX removed"

# ── 3. MetalLB ───────────────────────────────────────────────────────────────
log "Removing MetalLB..."
kubectl delete -f metallb-pool.yaml --ignore-not-found 2>/dev/null || true
kubectl delete -f https://raw.githubusercontent.com/metallb/metallb/v0.16.0/config/manifests/metallb-native.yaml --ignore-not-found 2>/dev/null || true
kubectl delete namespace metallb-system --ignore-not-found
success "MetalLB removed"

# ── 4. Calico ─────────────────────────────────────────────────────────────────
log "Removing Calico (via Tigera operator)..."
kubectl delete installation default --ignore-not-found
kubectl delete -f https://raw.githubusercontent.com/projectcalico/calico/v3.32.0/manifests/tigera-operator.yaml --ignore-not-found 2>/dev/null || true
kubectl delete -f https://raw.githubusercontent.com/projectcalico/calico/v3.32.0/manifests/v1_crd_projectcalico_org.yaml --ignore-not-found 2>/dev/null || true
kubectl delete namespace calico-system --ignore-not-found
kubectl delete namespace calico-apiserver --ignore-not-found
kubectl delete namespace tigera-operator --ignore-not-found

# Clean up any leftover CRDs
kubectl get crd | grep -E "calico|tigera|projectcalico" | awk '{print $1}' | \
  xargs -r kubectl delete crd --ignore-not-found

success "Calico removed"

# ── 5. Reset kube-proxy to iptables mode ─────────────────────────────────────
log "Resetting kube-proxy configmap to default iptables mode..."
kubectl get configmap kube-proxy -n kube-system -o yaml | \
  sed 's/mode: "ipvs"/mode: "iptables"/' | \
  kubectl apply -f - 2>/dev/null || warn "Could not reset kube-proxy config"
kubectl rollout restart daemonset/kube-proxy -n kube-system
success "kube-proxy reset"

echo ""
echo "═══════════════════════════════════════════════════════════"
echo "✅ Cleanup complete. Wait 30s before running install scripts."
echo "═══════════════════════════════════════════════════════════"
OOF

In [ ]:
chmod +x cleanup.sh && ./cleanup.sh

---

Script 3: Install Calico

In [ ]:
cat > install-calico.sh << 'OF'
#!/bin/bash
# =============================================================================
# Calico Install Script — v3.32.0 via Tigera Operator
# Fix included: MTU=1450, VXLAN encapsulation, kube-proxy IPVS + strictARP
# =============================================================================
set -euo pipefail

GREEN='\033[0;32m'; BLUE='\033[0;34m'; YELLOW='\033[1;33m'; NC='\033[0m'
log()     { echo -e "${BLUE}[INFO]${NC} $1"; }
success() { echo -e "${GREEN}[SUCCESS]${NC} $1"; }
warn()    { echo -e "${YELLOW}[WARN]${NC} $1"; }

POD_CIDR="10.244.0.0/16"

# ── 1. kube-proxy: IPVS + strictARP ─────────────────────────────────────────
log "Configuring kube-proxy for IPVS + strictARP..."
kubectl get configmap kube-proxy -n kube-system -o yaml | \
  sed 's/mode: ""/mode: "ipvs"/' | \
  sed 's/strictARP: false/strictARP: true/' | \
  kubectl apply -f -
kubectl rollout restart daemonset/kube-proxy -n kube-system
kubectl rollout status daemonset/kube-proxy -n kube-system --timeout=120s
success "kube-proxy configured"

# ── 2. Install Tigera Operator ────────────────────────────────────────────────
log "Installing Tigera operator..."
kubectl apply -f https://raw.githubusercontent.com/projectcalico/calico/v3.32.0/manifests/tigera-operator.yaml
log "Waiting for Tigera operator to be ready..."
kubectl rollout status deployment/tigera-operator -n tigera-operator --timeout=120s
success "Tigera operator ready"

# ── 3. Wait for CRDs ─────────────────────────────────────────────────────────
log "Waiting for Calico CRDs to be registered..."
until kubectl get crd installations.operator.tigera.io &>/dev/null; do
  echo "  not ready yet, waiting 5s..."; sleep 5
done
sleep 5
success "Calico CRDs registered"

# ── 4. Apply Installation CR ──────────────────────────────────────────────────
log "Applying Calico installation with VXLAN + MTU 1450..."
cat << YAML | kubectl apply -f -
apiVersion: operator.tigera.io/v1
kind: Installation
metadata:
  name: default
spec:
  variant: Calico
  calicoNetwork:
    mtu: 1450
    bgp: Disabled
    ipPools:
    - name: default-ipv4-ippool
      cidr: ${POD_CIDR}
      blockSize: 26
      encapsulation: VXLAN
      natOutgoing: Enabled
      nodeSelector: all()
---
apiVersion: operator.tigera.io/v1
kind: APIServer
metadata:
  name: default
spec: {}
YAML
success "Installation CR applied"

# ── 5. Wait for calico-system namespace ──────────────────────────────────────
log "Waiting for calico-system namespace to be created by operator..."
until kubectl get namespace calico-system &>/dev/null; do
  echo "  not ready yet, waiting 5s..."; sleep 5
done
success "calico-system namespace ready"

# ── 6. Wait for Calico nodes ──────────────────────────────────────────────────
log "Waiting for Calico nodes to be ready (this takes ~2 minutes)..."
until kubectl get daemonset calico-node -n calico-system &>/dev/null; do
  echo "  daemonset not created yet, waiting 5s..."; sleep 5
done
kubectl rollout status daemonset/calico-node -n calico-system --timeout=600s
success "Calico nodes ready"

log "Waiting for Calico API server..."
kubectl rollout status deployment/calico-apiserver -n calico-system --timeout=300s 2>/dev/null || \
  warn "Calico API server not yet ready — may take a few more minutes"
# ── 7. Apply Felix configuration ──────────────────────────────────────────────
log "Applying Felix configuration..."
cat << YAML | kubectl apply -f -
apiVersion: crd.projectcalico.org/v1
kind: FelixConfiguration
metadata:
  name: default
spec:
  genericXDPEnabled: false
  natPortRange: "32768:65535"
YAML
success "Felix configuration applied"

echo ""
echo "═══════════════════════════════════════════════════════════"
echo "✅ Calico installed successfully"
echo "   Pod CIDR : ${POD_CIDR}"
echo "   MTU      : 1450"
echo "   Mode     : VXLAN"
echo "═══════════════════════════════════════════════════════════"
echo "Run: kubectl get pods -n calico-system"
OF

In [ ]:
chmod +x install-calico.sh && ./install-calico.sh

---

Script 4: Install MetalLB

In [ ]:
cat > install-metallb.sh << 'OF'
#!/bin/bash
# =============================================================================
# MetalLB Install Script — v0.16.0
# =============================================================================
set -euo pipefail

GREEN='\033[0;32m'; BLUE='\033[0;34m'; NC='\033[0m'
log()     { echo -e "${BLUE}[INFO]${NC} $1"; }
success() { echo -e "${GREEN}[SUCCESS]${NC} $1"; }

LB_RANGE="172.16.6.90-172.16.6.95"

# ── 1. Install MetalLB ────────────────────────────────────────────────────────
log "Installing MetalLB v0.16.0..."
kubectl apply -f https://raw.githubusercontent.com/metallb/metallb/v0.16.0/config/manifests/metallb-native.yaml

log "Waiting for MetalLB controller..."
kubectl rollout status deployment/controller -n metallb-system --timeout=180s
kubectl rollout status daemonset/speaker -n metallb-system --timeout=180s
success "MetalLB ready"

# ── 2. Remove webhook (avoids timing issues on fresh install) ─────────────────
log "Removing MetalLB webhook..."
kubectl delete validatingwebhookconfiguration metallb-webhook-configuration --ignore-not-found
sleep 5

# ── 3. Apply IP pool + L2 advertisement ──────────────────────────────────────
log "Configuring IP address pool: ${LB_RANGE}..."
cat << EOF | kubectl apply -f -
apiVersion: metallb.io/v1beta1
kind: IPAddressPool
metadata:
  name: first-pool
  namespace: metallb-system
spec:
  addresses:
  - ${LB_RANGE}
---
apiVersion: metallb.io/v1beta1
kind: L2Advertisement
metadata:
  name: l2-adv
  namespace: metallb-system
spec:
  ipAddressPools:
  - first-pool
EOF
success "MetalLB IP pool configured"

echo ""
echo "═══════════════════════════════════════════════════════════"
echo "✅ MetalLB installed successfully"
echo "   IP Range : ${LB_RANGE}"
echo "═══════════════════════════════════════════════════════════"
echo "Run: kubectl get pods -n metallb-system"
OF

In [ ]:
chmod +x install-metallb.sh && ./install-metallb.sh

---

Script 5: Install Ingress-NGINX

In [ ]:
cat > install-ingress.sh << 'OF'
#!/bin/bash
# =============================================================================
# Ingress-NGINX Install Script
# =============================================================================
set -euo pipefail

GREEN='\033[0;32m'; BLUE='\033[0;34m'; NC='\033[0m'
log()     { echo -e "${BLUE}[INFO]${NC} $1"; }
success() { echo -e "${GREEN}[SUCCESS]${NC} $1"; }

LB_IP="172.16.6.90"

# ── 1. Add Helm repo ──────────────────────────────────────────────────────────
log "Adding ingress-nginx Helm repo..."
helm repo add ingress-nginx https://kubernetes.github.io/ingress-nginx
helm repo update

# ── 2. Install ────────────────────────────────────────────────────────────────
log "Installing ingress-nginx with DaemonSet + hostNetwork..."
cat << EOF > /tmp/nginx-ingress-values.yaml
controller:
  kind: DaemonSet
  hostNetwork: true
  dnsPolicy: ClusterFirstWithHostNet
  hostPort:
    enabled: true
    ports:
      http: 80
      https: 443
  service:
    type: LoadBalancer
    loadBalancerIP: ${LB_IP}
  admissionWebhooks:
    enabled: false
  metrics:
    enabled: true
  podDisruptionBudget:
    enabled: true
  tolerations:
  - key: "node-role.kubernetes.io/control-plane"
    operator: "Exists"
    effect: "NoSchedule"
  config:
    proxy-connect-timeout: "60"
    proxy-read-timeout: "60"
    proxy-send-timeout: "60"
EOF

helm upgrade --install ingress-nginx ingress-nginx/ingress-nginx \
  --namespace ingress-nginx \
  --create-namespace \
  -f /tmp/nginx-ingress-values.yaml

log "Waiting for ingress-nginx DaemonSet..."
kubectl rollout status daemonset/ingress-nginx-controller -n ingress-nginx --timeout=180s
success "Ingress-NGINX ready"

echo ""
echo "═══════════════════════════════════════════════════════════"
echo "✅ Ingress-NGINX installed successfully"
echo "   LoadBalancer IP : ${LB_IP}"
echo "═══════════════════════════════════════════════════════════"
echo "Run: kubectl get svc -n ingress-nginx"
OF

In [ ]:
chmod +x install-ingress.sh && ./install-ingress.sh

---

Script 6: Install Headlamp

In [ ]:
cat > install-headlamp.sh <<OF
#!/bin/bash
# =============================================================================
# Headlamp Install Script
# Fix included: ClusterRoleBinding so headlamp can reach the API server
# =============================================================================
set -euo pipefail

GREEN='\033[0;32m'; BLUE='\033[0;34m'; NC='\033[0m'
log()     { echo -e "${BLUE}[INFO]${NC} $1"; }
success() { echo -e "${GREEN}[SUCCESS]${NC} $1"; }

DOMAIN="headlamp.voip.local"

# ── 1. Add Helm repo ──────────────────────────────────────────────────────────
log "Adding headlamp Helm repo..."
helm repo add headlamp https://kubernetes-sigs.github.io/headlamp/
helm repo update

# ── 2. Install Headlamp ───────────────────────────────────────────────────────
log "Installing Headlamp..."
cat << EOF > /tmp/headlamp-values.yaml
ingress:
  enabled: true
  ingressClassName: nginx
  annotations:
    nginx.ingress.kubernetes.io/ssl-redirect: "false"
    nginx.ingress.kubernetes.io/proxy-connect-timeout: "60"
    nginx.ingress.kubernetes.io/proxy-read-timeout: "60"
    nginx.ingress.kubernetes.io/proxy-send-timeout: "60"
  hosts:
    - host: ${DOMAIN}
      paths:
        - path: /
          type: Prefix
EOF

helm upgrade --install my-headlamp headlamp/headlamp \
  -f /tmp/headlamp-values.yaml \
  --namespace headlamp \
  --create-namespace

# ── 3. Fix RBAC — this was missing and caused 504 ────────────────────────────
log "Applying ClusterRoleBinding for headlamp service account..."
kubectl create clusterrolebinding headlamp-admin \
  --clusterrole=cluster-admin \
  --serviceaccount=headlamp:my-headlamp \
  --dry-run=client -o yaml | kubectl apply -f -
success "RBAC configured"

# ── 4. Wait and verify ────────────────────────────────────────────────────────
log "Waiting for headlamp pod..."
kubectl rollout status deployment/my-headlamp -n headlamp --timeout=120s
success "Headlamp ready"

# ── 5. Generate access token ──────────────────────────────────────────────────
echo ""
echo "═══════════════════════════════════════════════════════════"
echo "✅ Headlamp installed successfully"
echo "   URL    : http://${DOMAIN}"
echo "═══════════════════════════════════════════════════════════"
echo ""
echo "🔑 Access token (save this):"
kubectl create token my-headlamp --namespace headlamp
echo ""
echo "Test access:"
echo "  curl -H 'Host: ${DOMAIN}' http://172.16.6.90"
OF

In [ ]:
chmod +x install-headlamp.sh  && ./install-headlamp.sh 

In [ ]:
# All 5 nodes in one shot
for node in 172.16.6.66 172.16.6.67 172.16.6.62 172.16.6.68 172.16.6.54; do
  echo "=== Fixing $node ==="
  ssh root@$node "echo 2 > /proc/sys/net/ipv4/conf/ens192/rp_filter && \
    echo 2 > /proc/sys/net/ipv4/conf/all/rp_filter && \
    cat > /etc/sysctl.d/99-kubernetes.conf << 'EOF'
net.ipv4.conf.all.rp_filter = 2
net.ipv4.conf.default.rp_filter = 2
net.ipv4.conf.ens192.rp_filter = 2
net.bridge.bridge-nf-call-iptables = 1
net.bridge.bridge-nf-call-ip6tables = 1
net.ipv4.ip_forward = 1
EOF
sysctl -p /etc/sysctl.d/99-kubernetes.conf && echo DONE"
done

In [ ]:
curl -H 'Host: headlamp.voip.local' http://172.16.6.90

In [ ]:
# Try crd.projectcalico.org/v1 instead
cat << EOF | kubectl apply -f -
apiVersion: crd.projectcalico.org/v1
kind: FelixConfiguration
metadata:
  name: default
spec:
  bpfEnabled: false
  chainInsertMode: Append
  bpfConnectTimeLoadBalancingEnabled: false
  natPortRange: "32768:65535"
  allowIPIPPacketsFromWorkloads: true
  allowVXLANPacketsFromWorkloads: true
EOF

In [ ]:
cat << EOF | kubectl apply -f -
apiVersion: crd.projectcalico.org/v1
kind: GlobalNetworkPolicy
metadata:
  name: allow-host-to-pods
spec:
  order: 0
  selector: all()
  ingress:
  - action: Allow
    source:
      nets:
      - 172.16.6.0/24
  egress:
  - action: Allow
EOF